In [1]:
import os 
import pandas as pd 
import numpy as np

In [2]:
df= pd.read_csv('/home/rajesh/work/ornella/unique_combination_cleaned.csv')

In [4]:
df.shape

(47696, 17)

In [5]:
df.isna().sum()

Unnamed: 0            0
Date Collected        0
Date Received         0
CowID                 0
Date Sent             0
Date Approved         0
ClientID              0
ProjectNo             0
Test                  0
Result                0
S-N Value             0
DIM                   0
DSLB                  0
combined_cow          0
date_collected        0
breeding_date         0
Unnamed: 16       47693
dtype: int64

In [6]:
# To see the total count of NaT values:
print(f"Total NaT values: {df['breeding_date'].isna().sum()}")

# To view all rows where breeding_date is NaT:
nat_rows = df[df['breeding_date'].isna()]

# Display the first few rows of the NaT records
print(nat_rows)


Total NaT values: 0
Empty DataFrame
Columns: [Unnamed: 0, Date Collected, Date Received, CowID, Date Sent, Date Approved, ClientID, ProjectNo, Test, Result, S-N Value, DIM, DSLB, combined_cow, date_collected, breeding_date, Unnamed: 16]
Index: []


In [13]:
df.columns

Index(['Unnamed: 0', 'Date Collected', 'Date Received', 'CowID', 'Date Sent',
       'Date Approved', 'ClientID', 'ProjectNo', 'Test', 'Result', 'S-N Value',
       'DIM', 'DSLB', 'combined_cow', 'date_collected', 'breeding_date',
       'Unnamed: 16'],
      dtype='object')

In [3]:
# 1. Strip the extra spaces, then 2. Convert to datetime
df["breeding_date"] = pd.to_datetime(df["breeding_date"].astype(str).str.strip(), format='mixed')
df['year']= df['breeding_date'].dt.year
df['month']= df['breeding_date'].dt.month
df['day']= df['breeding_date'].dt.day
df['month_day']= df['breeding_date'].dt.strftime('%m-%d')
# df = df.sort_values(["combined_cow", "breeding_date"])
# #Compute difference:
# df["prev_breed_date"] = df.groupby("combined_cow")["breeding_date"].shift()
# df["breed_diff_days"] = (df["breeding_date"] - df["prev_breed_date"]).dt.days
# #Flag abnormal:
# df["abnormal_breeding_interval"] = df["breed_diff_days"].between(1,2)
# #Round down:
# df.loc[df["abnormal_breeding_interval"], "breeding_date"] = df["prev_breed_date"]


In [ ]:
temp=[]
for combined_cow, data_cow in df.groupby(['combined_cow', 'year','month']):
    data_cow['date_diff']= data_cow['day'].diff()
    is_new_event = (data_cow['date_diff'] >= 2).fillna(True)
    data_cow['check_new_event']= is_new_event
    data_cow['cluster'] = is_new_event.cumsum()
    unique_cluster= len(data_cow['cluster'].unique())
    if unique_cluster>1:

        for cluster, data_cluster in data_cow.groupby('cluster'):
            adj_bred_final= data_cluster['breeding_date'].sort_values(ascending=True).iloc[0]
            data_cluster['adj_final_breeding']= adj_bred_final
            data_cluster['breed_event_id']= str(combined_cow[0])+'_'+str(combined_cow[1])+'_'+str(combined_cow[2])+'_'+str(cluster)
            temp.append(data_cluster)
    else :
        adj_bred_final= data_cow['breeding_date'].sort_values(ascending=True).iloc[0]
        data_cow['adj_final_breeding']= adj_bred_final
        data_cow['breed_event_id']= str(combined_cow[0])+'_'+str(combined_cow[1])+'_'+str(combined_cow[2])
        temp.append(data_cow)


        

In [54]:
import pandas as pd

# 1. Create a true datetime column. 
# This handles leap years, varying month lengths, and month/year rollovers automatically.
df['full_date'] = pd.to_datetime(df[['year', 'month', 'day']])

# 2. Sort by cow and date to ensure diff() calculates chronologically
df = df.sort_values(by=['combined_cow', 'full_date']).reset_index(drop=True)

# 3. Calculate difference in days strictly within each cow's history
df['date_diff'] = df.groupby('combined_cow')['full_date'].diff().dt.days

# 4. A new event is triggered if the gap is >= 2 days, or if it's the very first record (NaN)
df['check_new_event'] = (df['date_diff'] >= 10) | df['date_diff'].isna()

# 5. Create a global cluster ID for each cow using cumsum()
df['cluster'] = df.groupby('combined_cow')['check_new_event'].cumsum()

# 6. Assign the earliest breeding date per cluster
df['adj_final_breeding'] = df.groupby(['combined_cow', 'cluster'])['breeding_date'].transform('min')

# 7. Create the breed_event_id. 
# Since an event could start on March 31 and end on April 1, we should base 
# the ID on the year and month the cluster *started*.
cluster_starts = df.groupby(['combined_cow', 'cluster'])['full_date'].transform('min')
df['event_year'] = cluster_starts.dt.year.astype(str)
df['event_month'] = cluster_starts.dt.month.astype(str)

# Build the ID string: cow_year_month_cluster
df['breed_event_id'] = (
    df['combined_cow'].astype(str) + '_' + 
    df['event_year'] + '_' + 
    df['event_month'] + '_' + 
    df['cluster'].astype(str)
)

# Optional: Clean up helper columns
df = df.drop(columns=['event_year', 'event_month'])

In [55]:
df[df['combined_cow']=='10328.0401430.0'].sort_values('breeding_date', ascending=False)[['breeding_date', 'adj_final_breeding', 'day', 'month', 'year', 'breed_event_id', 'Result', 'date_collected']]

,breeding_date,adj_final_breeding,day,month,year,breed_event_id,Result,date_collected
742,2025-04-03,2025-03-27,3,4,2025,10328.0401430.0_2025_3_4,Pregnant,5/8/2025
741,2025-03-27,2025-03-27,27,3,2025,10328.0401430.0_2025_3_4,Pregnant,6/19/2025
740,2025-02-10,2025-02-10,10,2,2025,10328.0401430.0_2025_2_3,Open,3/20/2025
739,2022-12-10,2022-12-09,10,12,2022,10328.0401430.0_2022_12_2,Pregnant,1/20/2023
738,2022-12-09,2022-12-09,9,12,2022,10328.0401430.0_2022_12_2,Pregnant,2/24/2023
737,2021-11-18,2021-11-18,18,11,2021,10328.0401430.0_2021_11_1,Pregnant,1/28/2022


In [9]:
data_cow= pd.concat(temp)

In [160]:
str(combined_cow[0])+'_'+str(combined_cow[1])+'_'+str(combined_cow[2])

'999999.0401430.0_2025_7'

In [14]:
data_final[data_final['combined_cow']=='15774.0401430.0'].sort_values('breeding_date', ascending=False)[['breeding_date', 'adj_final_breeding', 'day', 'month', 'year', 'breed_event_id']]

,breeding_date,adj_final_breeding,day,month,year,breed_event_id
18862,2025-08-07,2025-08-07,7,8,2025,15774.0401430.0_2025_8_3
18860,2024-08-17,2024-08-17,17,8,2024,15774.0401430.0_2024_8_2
18861,2024-08-17,2024-08-17,17,8,2024,15774.0401430.0_2024_8_2
18859,2023-06-01,2023-05-31,1,6,2023,15774.0401430.0_2023_5_1
18858,2023-05-31,2023-05-31,31,5,2023,15774.0401430.0_2023_5_1


In [56]:
temp_df=[]
for cow_id, data_id in df.groupby('combined_cow'):
    unique_breeding_event= sorted(data_id['adj_final_breeding'].unique())
    print(unique_breeding_event)
    dict_event={value:index+1 for index,value in enumerate(unique_breeding_event)}
    data_id['ai_id']= data_id['adj_final_breeding'].apply(lambda x: dict_event[x])
    temp_df.append(data_id)
    

[Timestamp('2022-01-28 00:00:00')]
[Timestamp('2022-12-09 00:00:00'), Timestamp('2023-11-23 00:00:00')]
[Timestamp('2021-12-03 00:00:00')]
[Timestamp('2022-08-11 00:00:00'), Timestamp('2022-09-19 00:00:00'), Timestamp('2022-12-10 00:00:00')]
[Timestamp('2022-05-27 00:00:00'), Timestamp('2023-05-17 00:00:00')]
[Timestamp('2022-01-14 00:00:00'), Timestamp('2022-03-16 00:00:00')]
[Timestamp('2021-11-04 00:00:00'), Timestamp('2022-10-20 00:00:00')]
[Timestamp('2022-02-03 00:00:00'), Timestamp('2022-03-11 00:00:00'), Timestamp('2022-04-22 00:00:00'), Timestamp('2022-06-03 00:00:00'), Timestamp('2023-08-14 00:00:00'), Timestamp('2024-08-12 00:00:00')]
[Timestamp('2022-02-24 00:00:00'), Timestamp('2023-02-16 00:00:00'), Timestamp('2023-05-13 00:00:00'), Timestamp('2025-05-21 00:00:00')]
[Timestamp('2022-08-18 00:00:00'), Timestamp('2023-08-10 00:00:00'), Timestamp('2024-11-11 00:00:00'), Timestamp('2024-12-22 00:00:00'), Timestamp('2025-02-13 00:00:00'), Timestamp('2025-05-08 00:00:00')]
[Tim

In [57]:
data_final= pd.concat(temp_df)

In [79]:
temp_sample=[]
for id, data_id in data_final.groupby(['combined_cow', 'ai_id']):
    data_id['date_collected']= pd.to_datetime(data_id['date_collected'])
    data_id= data_id.sort_values('date_collected', ascending= True)
    unique_sample_event= data_id['date_collected'].unique()
    dict_event={value:index+1 for index,value in enumerate(unique_sample_event)}
    data_id['sample_id']= data_id['date_collected'].apply(lambda x: dict_event[x])
    temp_sample.append(data_id)

In [80]:

data_final_sample= pd.concat(temp_sample)

In [81]:
temp_class=[]
for bred, data_bred in data_final_sample.groupby('breed_event_id'):
    max_event= max(data_bred['sample_id'])
    data_bred['sample_class']= data_bred['sample_id'].apply(lambda x: 'first' if x==1 else ('last' if x==max_event else 'intermediate'))
    temp_class.append(data_bred)
    

In [96]:
class_df= pd.concat(temp_class)

In [97]:
class_df.columns

Index(['Unnamed: 0', 'Date Collected', 'Date Received', 'CowID', 'Date Sent',
       'Date Approved', 'ClientID', 'ProjectNo', 'Test', 'Result', 'S-N Value',
       'DIM', 'DSLB', 'combined_cow', 'date_collected', 'breeding_date',
       'Unnamed: 16', 'year', 'month', 'day', 'month_day', 'full_date',
       'date_diff', 'check_new_event', 'cluster', 'adj_final_breeding',
       'breed_event_id', 'ai_id', 'sample_id', 'sample_class'],
      dtype='object')

In [98]:
class_df.to_csv('data_with_sample_class.csv')

In [17]:
import pandas as pd 
import os 
import numpy as np 

class_df= pd.read_csv('data_with_sample_class.csv')

# condition time

In [25]:
class_df['Result'].value_counts()

Result
Pregnant    36213
Open         8134
Re-Check     3349
Name: count, dtype: int64

In [ ]:
temp_bred_data=[]
for bred, data_bred in class_df.groupby('breed_event_id'):
    if len(data_bred)==1:
        data_bred['final_resolved']= data_bred['Result'].iloc[0]
    
    elif:
        len(data_bred)==2:
        unique_samples = data_bred['Result'].unique()
        if len(unique_samples) == 1 and unique_samples[0] == 'Pregnant':
            data_bred['final_resolved'] = 'Pregnant'
       
    elif:
         len(data_bred)==2:
         samples= list(data_bred['Result'])
         if samples[0]=='Pregnant' & samples[1]=='Open':
            data_bred['final_resolved']=='loss'

    elif:

        len(data_bred)>2:
        samples= list(data_bred['Result'])

        if samples[0]=='Pregnant':
            samples_less= samples.pop(samples[0]):
            if 'Re-Check' in samples_less:
                last_sample= samples[-1]
                if last_sample=='Pregnant':
                    data_bred['final_resolved']='Pregant_with_rc'

                if last_sample=='Open':

                    data_bred['final_resolved']=='Preg_loss_with_rc'

    elif:
        len(data_bred)=2:
        samples= list(data_bred['Result'])

        if samples[0]=='Re-Check':
            samples_less= samples.pop(samples[0]):
           
                last_sample= samples[-1]
                if last_sample=='Open':
                    data_bred['final_resolved']='rc_open_2'

    elif:
        len(data_bred)>2:
        samples= list(data_bred['Result'])

        if samples[0]=='Re-Check':
            samples_less= samples.pop(samples[0]):
            if 'Pregnant' in samples_less:
                last_sample= samples[-1]
                if last_sample=='Open':
                    data_bred['final_resolved']='rc_preg_loss'

    elif:
        len(data_bred)>2:
        samples= list(data_bred['Result'])

        if samples[0]=='Re-Check':
            samples_less= samples.pop(samples[0]):
            if 'Re-Check' in samples_less:
                last_sample= samples[-1]
                if last_sample=='Open':
                    data_bred['final_resolved']='rc_open'

    else:
        len(data_bred)>2:
        samples= list(data_bred['Result'])

        if samples[0]=='Re-Check':
            samples_less= samples.pop(samples[0]):
            if 'Re-Check' in samples_less:
                last_sample= samples[-1]
                if last_sample=='Pregnant':
                    data_bred['final_resolved']='rc_preg'

        temp_bred_data.append(data_bred)

                        

            



['Pregnant']
['Pregnant']


In [99]:
import pandas as pd

temp_bred_data = []

for bred, data_bred in class_df.groupby('breed_event_id'):
    # Create a copy to avoid Pandas "SettingWithCopyWarning"
    data_bred = data_bred.copy()
    data_bred= data_bred.sort_values('sample_id', ascending=True)
    
    # Store length and convert results to a list for easier indexing
    n_samples = len(data_bred)
    samples = data_bred['Result'].tolist()
    
    # Default value in case a scenario isn't explicitly caught
    final_resolved = None 
    
    if n_samples == 1:
        final_resolved = samples[0]
        
    elif n_samples == 2:
        if samples[0] == 'Pregnant' and samples[1] == 'Pregnant':
            final_resolved = 'Pregnant'
        elif samples[0] == 'Pregnant' and samples[1] == 'Open':
            final_resolved = 'loss'
        elif samples[0] == 'Re-Check' and samples[1] == 'Open':
            final_resolved = 'rc_open_2'
        elif samples[0] == 'Re-Check' and samples[1] == 'Pregnant':
            final_resolved = 'rc_preg_2'
            
    elif n_samples > 2:
        first_sample = samples[0]
        last_sample = samples[-1]
        samples_less = samples[1:-1] 
        
        if first_sample == 'Pregnant':
            if 'Re-Check' in samples_less:
                if last_sample == 'Pregnant':
                    final_resolved = 'Pregant_with_rc'
                elif last_sample == 'Open':
                    final_resolved = 'Preg_loss_with_rc'
                    
        elif first_sample == 'Re-Check':
            if 'Pregnant' in samples_less:
                if last_sample == 'Open':
                    final_resolved = 'rc_preg_loss'
            elif 'Re-Check' in samples_less:
                if last_sample == 'Open':
                    final_resolved = 'rc_open'
                elif last_sample == 'Pregnant':
                    final_resolved = 'rc_preg'
    
    # Assign the calculated outcome to the whole group
    data_bred['final_resolved'] = final_resolved
    
    # Append MUST be outside the if/elif blocks so every group is saved
    temp_bred_data.append(data_bred)

# Recombine everything into a single dataframe at the end
final_df = pd.concat(temp_bred_data, ignore_index=True)

In [100]:
final_df[final_df['combined_cow']=='10841.0401430.0'].sort_values('breeding_date', ascending=False)[['breeding_date', 'adj_final_breeding', 'day', 'month', 'year', 'breed_event_id', 'Result', 'final_resolved', 'DSLB', 'ai_id', 'sample_id', 'date_collected']]

,breeding_date,adj_final_breeding,day,month,year,breed_event_id,Result,final_resolved,DSLB,ai_id,sample_id,date_collected
2019,2023-05-15,2023-05-14,15,5,2023,10841.0401430.0_2023_5_2,Re-Check,Preg_loss_with_rc,74,2,2,2023-07-28
2020,2023-05-15,2023-05-14,15,5,2023,10841.0401430.0_2023_5_2,Re-Check,Preg_loss_with_rc,81,2,3,2023-08-04
2018,2023-05-14,2023-05-14,14,5,2023,10841.0401430.0_2023_5_2,Pregnant,Preg_loss_with_rc,40,2,1,2023-06-23
2021,2023-05-14,2023-05-14,14,5,2023,10841.0401430.0_2023_5_2,Open,Preg_loss_with_rc,89,2,4,2023-08-11
2016,2023-03-31,2023-03-30,31,3,2023,10841.0401430.0_2023_3_1,Re-Check,None,35,1,1,2023-05-05
2017,2023-03-30,2023-03-30,30,3,2023,10841.0401430.0_2023_3_1,Re-Check,None,43,1,2,2023-05-12


In [29]:
final_df[final_df['combined_cow']=='17058.0401430.0'].sort_values('breeding_date', ascending=False)[['combined_cow', 'ai_id', 'breed_event_id']][['breed_event_id']].iloc[0]

breed_event_id    0             1.0392159.0\n1             1.039...
Name: 21840, dtype: object

In [35]:
import pandas as pd 
import os 
final_df= pd.read_csv('2_27_data.csv')

In [ ]:
temp_df=[]
final_df['adj_final_breeding']=  pd.to_datetime(final_df['adj_final_breeding'])
for cow_id, data_id in final_df.groupby('combined_cow'):
    unique_dates = sorted(data_id['adj_final_breeding'].unique())
    
    # Calculate gaps
    gap_dates = {unique_dates[i]: (unique_dates[i+1] - unique_dates[i]).days 
                 for i in range(len(unique_dates) - 1)}
    
    # Filter for gaps < 300 days
    selected_gap = {date: gap for date, gap in gap_dates.items() if gap <300}
    for adj_date in selected_gap.keys():
        # Get data for this specific date
        data_adj_date = data_id[data_id['adj_final_breeding'] == adj_date].copy()
        data_adj_date= data_adj_date.sort_values('sample_id', ascending=True)
        if len(data_adj_date) > 1:
            print(data_adj_date)
            results_list = data_adj_date['Result'].tolist()
            last_sample = results_list[-1]

            # 1. Skip if the last status of the day was 'Open'
            if last_sample == 'Open':
                continue
            
            # 2. Check for Pregnancy Loss / Re-breeding
            if all(res=="Pregnant" for  res in results_list) & all(res!='Re-Check' for res in results_list):
                # Use .loc to avoid the "SettingWithCopy" warning
                final_df.loc[data_adj_date.index, 'final_resolved'] = 'preg_loss_rebred'

            if any(res=="Pregnant" for  res in results_list) & any(res!='Re-Check' for res in results_list):
                # Use .loc to avoid the "SettingWithCopy" warning
                final_df.loc[data_adj_date.index, 'final_resolved'] = 'rc_preg_loss_rebred'
            # 3. Check if EVERYTHING that day was a 'Re-Check'
            if all(res == 'Re-Check' for res in results_list):
                final_df.loc[data_adj_date.index, 'final_resolved'] = 'rc_rebred'




    Unnamed: 0 Date Collected   Date Received  CowID        Date Sent  \
21        2919       4/8/2022  4/9/2022 12:57  10008   4/8/2022 22:06   
22        3217      4/15/2022  4/16/2022 9:41  10008  4/15/2022 21:01   

      Date Approved  ClientID  ProjectNo                   Test    Result  \
21   4/9/2022 17:58    401430   22040011  Pregnancy Test - Milk  Re-Check   
22  4/16/2022 16:20    401430   22040021  Pregnancy Test - Milk      Open   

    ...  full_date  date_diff  check_new_event cluster adj_final_breeding  \
21  ... 2022-03-11       36.0             True       2         2022-03-11   
22  ... 2022-03-11        0.0            False       2         2022-03-11   

              breed_event_id  ai_id  sample_id  sample_class  final_resolved  
21  10008.0401430.0_2022_3_2      2          1         first       rc_open_2  
22  10008.0401430.0_2022_3_2      2          2          last       rc_open_2  

[2 rows x 31 columns]
    Unnamed: 0 Date Collected    Date Received  CowID   

In [105]:
final_df.to_csv('3_9_2026.csv')

In [114]:
for id, cow_data in final_df.groupby('combined_cow'):
    if len(cow_data['ai_id'].unique())==1:
        cow_data=  cow_data.sort_values('sample_id',ascending= True)
        final_result= cow_data['Result'].values[-1]

        if final_result=='Re-Check':
            final_df.loc[cow_data.index, 'final_resolved']== 'rc_unresolved'
            

        if final_result=='Open' :
            if  any(res=='Pregnant' for res in cow_data['Result']):
                final_df.loc[cow_data.index, 'final_resolved']== 'loss_2_open'

    if len(cow_data['ai_id'].unique())>1:
        unique_dates = sorted(cow_data['adj_final_breeding'].unique())
        
        # Calculate gaps
        gap_dates = {unique_dates[i]: (unique_dates[i+1] - unique_dates[i]).days 
                    for i in range(len(unique_dates) - 1)}
        
        # Filter for gaps < 300 days
        selected_gap = {date: gap for date, gap in gap_dates.items() if gap >300}
        for adj_date in selected_gap.keys():
            # Get data for this specific date
            data_adj_date = cow_data[cow_data['adj_final_breeding'] == adj_date].copy()
            data_adj_date= data_adj_date.sort_values('sample_id', ascending=True)
            if len(data_adj_date) > 1:
                results_list = data_adj_date['Result'].tolist()
                last_sample = results_list[-1]


                if last_sample == 'Re-Check':
                    final_df.loc[data_adj_date.index, 'final_resolved'] = 'rc_unresolved_>_1_bred'
            

In [104]:
final_df[final_df['combined_cow']=='10017.0401430.0'][['Result', 'final_resolved', 'ai_id', 'sample_id', 'adj_final_breeding']]

,Result,final_resolved,ai_id,sample_id,adj_final_breeding
64,Pregnant,None,1,1,2022-02-18
65,Re-Check,None,1,2,2022-02-18
66,Re-Check,None,1,3,2022-02-18
67,Pregnant,Pregnant,2,1,2023-04-02
68,Pregnant,Pregnant,2,2,2023-04-02


In [117]:
final_df[final_df['final_resolved'].isna()]

,Unnamed: 0,Date Collected,Date Received,CowID,Date Sent,Date Approved,ClientID,ProjectNo,Test,Result,...,full_date,date_diff,check_new_event,cluster,adj_final_breeding,breed_event_id,ai_id,sample_id,sample_class,final_resolved
75,2565,4/1/2022,4/2/2022 11:10,10022,4/1/2022 22:43,4/2/2022 15:46,401430,22040001,Pregnancy Test - Milk,Pregnant,...,2022-03-03,NaN,True,1,2022-03-03,10022.0401430.0_2022_3_1,1,1,first,None
76,3803,5/6/2022,5/7/2022 10:30,10022,5/6/2022 20:01,5/7/2022 15:14,401430,22050005,Pregnancy Test - Milk,Pregnant,...,2022-03-03,0.0,False,1,2022-03-03,10022.0401430.0_2022_3_1,1,2,intermediate,None
77,4040,5/20/2022,5/21/2022 11:20,10022,5/20/2022 20:43,5/21/2022 17:11,401430,22050017,Pregnancy Test - Milk,Pregnant,...,2022-03-03,0.0,False,1,2022-03-03,10022.0401430.0_2022_3_1,1,3,last,None
163,4521,5/12/2023,5/13/2023 7:47,10069,5/13/2023 6:26,5/13/2023 12:34,401430,23050012,Pregnancy Test - Milk,Pregnant,...,2023-03-31,380.0,True,2,2023-03-31,10069.0401430.0_2023_3_2,2,1,first,None
164,5545,6/16/2023,6/17/2023 7:41,10069,6/16/2023 14:56,6/17/2023 12:11,401430,23060019,Pregnancy Test - Milk,Re-Check,...,2023-04-01,1.0,False,2,2023-03-31,10069.0401430.0_2023_3_2,2,2,last,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47663,8197,11/4/2022,11/5/2022 11:51,9998,11/4/2022 20:25,11/6/2022 0:45,401430,22110005,Pregnancy Test - Milk,Re-Check,...,2022-07-23,0.0,False,2,2022-07-22,9998.0401430.0_2022_7_2,2,7,intermediate,None
47664,8315,11/11/2022,11/12/2022 13:09,9998,11/11/2022 20:26,11/12/2022 17:25,401430,22110010,Pregnancy Test - Milk,Pregnant,...,2022-07-23,0.0,False,2,2022-07-22,9998.0401430.0_2022_7_2,2,8,last,None
47669,2616,4/1/2022,4/2/2022 11:08,9999,4/1/2022 22:43,4/2/2022 15:46,401430,22040001,Pregnancy Test - Milk,Pregnant,...,2022-03-01,40.0,True,2,2022-03-01,9999.0401430.0_2022_3_2,2,1,first,None
47670,3769,5/6/2022,5/7/2022 10:36,9999,5/6/2022 20:01,5/7/2022 15:14,401430,22050005,Pregnancy Test - Milk,Pregnant,...,2022-03-01,0.0,False,2,2022-03-01,9999.0401430.0_2022_3_2,2,2,intermediate,None


In [49]:
temp_list = []

final_df['adj_final_breeding'] = pd.to_datetime(final_df['adj_final_breeding'])

for cow_id, data_id in final_df.groupby('combined_cow'):
    
    unique_dates = sorted(data_id['adj_final_breeding'].unique())
    
    # Calculate gaps
    gap_dates = {
        unique_dates[i]: (unique_dates[i+1] - unique_dates[i]).days
        for i in range(len(unique_dates) - 1)
    }
    
    # Filter for gaps between 3 and 10 days
    selected_gap = {date: gap for date, gap in gap_dates.items() if 10 < gap < 11}
    
    if selected_gap:
        my_dict = {cow_id: selected_gap}
        temp_list.append(my_dict)

In [50]:
temp_list[0]

IndexError: list index out of range

In [48]:
final_df[final_df['final_resolved']=='Re-Check'].combined_cow.value_counts().reset_index()

,combined_cow,count
0,17058.0401430.0,2
1,1504.0392159.0,2
2,968.0401430.0,2
3,2350.0401430.0,2
4,8050.0401430.0,2
...,...,...
356,15179.0401430.0,1
357,15144.0401430.0,1
358,1509.0392159.0,1
359,15058.0401430.0,1


In [29]:
final_df.to_csv('3_2_data.csv')

In [23]:
data_adj_date[['Result', 'final_resolved', 'ai_id', 'adj_final_breeding', 'sample_id']]

,Result,final_resolved,ai_id,adj_final_breeding,sample_id
46673,Pregnant,loss,3,2023-04-25,1
46674,Open,loss,3,2023-04-25,2


In [24]:
data_adj_date['Result'].value_counts()

Result
Pregnant    1
Open        1
Name: count, dtype: int64

In [ ]:
for cow, data_cow in df.groupby('combined_cow'):
    test_str = list(data_cow['Result'].values)
    
    # First, make sure the cow actually has at least 2 results to avoid an IndexError
    if len(test_str) >= 2:
        # Check if both the first and second results are 'Re-Check'
        if test_str[0] == 'Re-Check' and test_str[1] == 'Re-Check':
            print(cow)
            

10076.0401430.0
10669.0401430.0
10695.0401430.0
10705.0401430.0
10727.0401430.0
10756.0401430.0
10799.0401430.0
10841.0401430.0
11036.0401430.0
11308.0401430.0
11362.0401430.0
11457.0401430.0
11579.0401430.0
11693.0401430.0
12053.0401430.0
12410.0401430.0
12464.0401430.0
12572.0401430.0
12608.0401430.0
1267.0401430.0
12806.0401430.0
12929.0401430.0
13345.0401430.0
13400.0401430.0
13606.0401430.0
13923.0401430.0
13968.0401430.0
14046.0401430.0
14059.0401430.0
14168.0401430.0
14497.0401430.0
14535.0401430.0
14538.0401430.0
14650.0401430.0
14891.0401430.0
15000.0401430.0
15020.0401430.0
15049.0401430.0
15420.0401430.0
15627.0401430.0
15846.0401430.0
15898.0401430.0
15956.0401430.0
15988.0401430.0
16556.0401430.0
1667.0401430.0
16887.0401430.0
17302.0401430.0
17346.0401430.0
19110.0401430.0
19139.0401430.0
19781.0401430.0
2237.0401430.0
24730.0392159.0
2514.0401430.0
2785.0401430.0
3223.0392159.0
3302.0401430.0
3480.0392159.0
3860.0392159.0
3890.0392159.0
3966.0401430.0
3995.0401430.0
4136

# ordering dataset

In [7]:
df = df.sort_values(
    ["combined_cow", "breeding_date", "date_collected"]
)


In [8]:
df = df.sort_values(
    ["combined_cow", "breeding_date", "date_collected"]
)


In [9]:
df["breeding_event_id"] = (
    df.groupby("combined_cow")["breeding_date"]
      .rank(method="dense")
      .astype(int)
)


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [1]:
import pandas as pd 
class_df=pd.read_csv('/home/rajesh/work/ornella/3_9_2026.csv')


temp_bred_data = []

# --- BLOCK 1: Resolve sequences by breed_event_id ---
for bred, data_bred in class_df.groupby('breed_event_id'):
    # Create a copy to avoid Pandas "SettingWithCopyWarning"
    data_bred = data_bred.copy()
    data_bred = data_bred.sort_values('sample_id', ascending=True)
    
    n_samples = len(data_bred)
    samples = data_bred['Result'].tolist()
    final_resolved = None 
    
    if n_samples == 1:
        final_resolved = samples[0]
        
    elif n_samples == 2:
        if samples[0] == 'Pregnant' and samples[1] == 'Pregnant':
            final_resolved = 'Pregnant'
        elif samples[0] == 'Pregnant' and samples[1] == 'Open':
            final_resolved = 'loss'
        elif samples[0] == 'Re-Check' and samples[1] == 'Open':
            final_resolved = 'rc_open_2'
        elif samples[0] == 'Re-Check' and samples[1] == 'Pregnant':
            final_resolved = 'rc_preg_2'
            
    elif n_samples > 2:
        first_sample = samples[0]
        last_sample = samples[-1]
        samples_less = samples[1:-1] 
        
        if first_sample == 'Pregnant':
            if 'Re-Check' in samples_less:
                if last_sample == 'Pregnant':
                    final_resolved = 'Pregant_with_rc'
                elif last_sample == 'Open':
                    final_resolved = 'Preg_loss_with_rc'
                    
        elif first_sample == 'Re-Check':
            if 'Pregnant' in samples_less:
                if last_sample == 'Open':
                    final_resolved = 'rc_preg_loss'
            elif 'Re-Check' in samples_less:
                if last_sample == 'Open':
                    final_resolved = 'rc_open'
                elif last_sample == 'Pregnant':
                    final_resolved = 'rc_preg'
    
    data_bred['final_resolved'] = final_resolved
    temp_bred_data.append(data_bred)

# Recombine everything into a single dataframe
final_df = pd.concat(temp_bred_data, ignore_index=True)

# --- BLOCK 2: Resolve rebreds and gaps by combined_cow ---
final_df['adj_final_breeding'] = pd.to_datetime(final_df['adj_final_breeding'])

for cow_id, cow_data in final_df.groupby('combined_cow'):
    
    # Scenario A: Only one AI event for this cow
    if len(cow_data['ai_id'].unique()) == 1:
        cow_data = cow_data.sort_values('sample_id', ascending=True)
        final_result = cow_data['Result'].values[-1]

        if final_result == 'Re-Check':
            # FIXED: Changed == to =
            final_df.loc[cow_data.index, 'final_resolved'] = 'rc_unresolved'
            
        elif final_result == 'Open':
            if 'Pregnant' in cow_data['Result'].values:
                # FIXED: Changed == to =
                final_df.loc[cow_data.index, 'final_resolved'] = 'loss_2_open'

    # Scenario B: Multiple AI events for this cow
    elif len(cow_data['ai_id'].unique()) > 1:
        unique_dates = sorted(cow_data['adj_final_breeding'].unique())
        
        # Calculate gaps between consecutive breeding dates
        gap_dates = {unique_dates[i]: (unique_dates[i+1] - unique_dates[i]).days 
                     for i in range(len(unique_dates) - 1)}
        
        for adj_date, gap in gap_dates.items():
            # Get data for this specific date
            data_adj_date = cow_data[cow_data['adj_final_breeding'] == adj_date].copy()
            data_adj_date = data_adj_date.sort_values('sample_id', ascending=True)
            
            if len(data_adj_date) > 1:
                results_list = data_adj_date['Result'].tolist()
                last_sample = results_list[-1]

                # Gaps LESS than 300 days
                if gap < 300:
                    if last_sample == 'Open':
                        continue
                    
                    # 1. All tests were Pregnant
                    if all(res == "Pregnant" for res in results_list):
                        final_df.loc[data_adj_date.index, 'final_resolved'] = 'preg_loss_rebred'

                    # 2. Mix of Pregnant and Re-Check
                    elif "Pregnant" in results_list and "Re-Check" in results_list:
                        final_df.loc[data_adj_date.index, 'final_resolved'] = 'rc_preg_loss_rebred'
                    
                    # 3. All tests were Re-Check
                    elif all(res == 'Re-Check' for res in results_list):
                        final_df.loc[data_adj_date.index, 'final_resolved'] = 'rc_rebred'

                # # Gaps GREATER than 300 days
                # elif gap > 300:
                #     if last_sample == 'Re-Check':
                #         final_df.loc[data_adj_date.index, 'final_resolved'] = 'rc_unresolved_>_1_bred'

In [28]:
import pandas as pd

# Load data and ensure datetime format
class_df = pd.read_csv('/home/rajesh/work/ornella/3_9_2026.csv')
class_df['adj_final_breeding'] = pd.to_datetime(class_df['adj_final_breeding'])

# Initialize the column to avoid any assignment issues later
class_df['final_resolved'] = None

def resolve_cow_history(cow_data):
    """
    Processes all breeding events for a single cow, resolving the pregnancy test 
    sequences and adjusting outcomes based on gaps between rebreds.
    """
    # Create a copy to avoid SettingWithCopyWarning, then sort chronologically
    cow_data = cow_data.copy()
    cow_data = cow_data.sort_values(['adj_final_breeding', 'sample_id'])
    
    # Identify unique breeding events for this cow in chronological order
    breed_events = cow_data['breed_event_id'].unique()
    n_events = len(breed_events)

    for i, event_id in enumerate(breed_events):
        mask = cow_data['breed_event_id'] == event_id
        event_df = cow_data[mask]
        
        samples = event_df['Result'].tolist()
        n_samples = len(samples)
        
        # Skip if no samples exist for this event
        if n_samples == 0: 
            continue

        first = samples[0]
        last = samples[-1]
        resolved = None

        # =======================================================
        # STEP 1: Base Resolution (Pattern-Matching)
        # =======================================================
        if n_samples == 1:
            resolved = first
        else:
            # 1. Sequences ending in PREGNANT
            if last == 'Pregnant':
                if first == 'Pregnant':
                    if 'Open' in samples: resolved = 'preg_open_preg'
                    elif 'Re-Check' in samples: resolved = 'Pregant_with_rc'
                    else: resolved = 'Pregnant'
                elif first == 'Re-Check':
                    resolved = 'rc_preg'
                elif first == 'Open':
                    resolved = 'open_to_preg'
                    
            # 2. Sequences ending in OPEN
            elif last == 'Open':
                if 'Pregnant' in samples: resolved = 'loss'
                elif first == 'Re-Check': resolved = 'rc_open'
                else: resolved = 'Open'
                    
            # 3. Sequences ending in RE-CHECK
            elif last == 'Re-Check':
                if 'Pregnant' in samples: resolved = 'preg_to_rc_unresolved'
                elif 'Open' in samples: resolved = 'open_to_rc_unresolved'
                else: resolved = 'rc_unresolved'

        # =======================================================
        # STEP 2: Sequence/Gap Overrides (Rebred Logic)
        # =======================================================
        if n_events == 1:
            # Scenario A: Only one AI event for this cow overall
            if last == 'Re-Check':
                resolved = 'rc_unresolved'
            elif last == 'Open' and 'Pregnant' in samples:
                resolved = 'loss_2_open'
        else:
            # Scenario B: Multiple AI events. Check the gap to the next event.
            if i < n_events - 1:
                curr_date = event_df['adj_final_breeding'].iloc[0]
                
                # Look ahead to the next breeding event date
                next_event_df = cow_data[cow_data['breed_event_id'] == breed_events[i+1]]
                next_date = next_event_df['adj_final_breeding'].iloc[0]
                gap = (next_date - curr_date).days

                # Overrides based on gap (skip override if last test was 'Open')
                if gap < 300 and n_samples > 1:
                    if last != 'Open':
                        if all(r == 'Pregnant' for r in samples):
                            resolved = 'preg_loss_rebred'
                        elif 'Pregnant' in samples and 'Re-Check' in samples:
                            resolved = 'rc_preg_loss_rebred'
                        elif all(r == 'Re-Check' for r in samples):
                            resolved = 'rc_rebred'
                            
                elif gap >= 300:
                    if last == 'Re-Check':
                        resolved = 'rc_unresolved_>_1_bred'

        # Assign the final computed status back to this specific event's rows
        cow_data.loc[mask, 'final_resolved'] = resolved

    return cow_data

# Apply the unified function to each cow
# group_keys=False prevents pandas from adding an extra index level
final_df = class_df.groupby('combined_cow', group_keys=False).apply(resolve_cow_history)

# Optional: verify your outputs
# print(final_df[['combined_cow', 'breed_event_id', 'Result', 'final_resolved']].head(20))

/tmp/ipykernel_262637/1283899811.py:107: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_df = class_df.groupby('combined_cow', group_keys=False).apply(resolve_cow_history)


In [29]:
# 1. Filter the dataframe for rows where final_resolved is missing (NaN or None)
unhandled_df = final_df[final_df['final_resolved'].isna()]

# 2. Group by the breeding event to see the exact sequence of results that were missed
unhandled_sequences = unhandled_df.groupby('breed_event_id')['Result'].apply(list).reset_index()
unhandled_sequences.rename(columns={'Result': 'unhandled_sequence'}, inplace=True)

# 3. Convert the lists to tuples so we can easily count the unique patterns
unhandled_sequences['sequence_tuple'] = unhandled_sequences['unhandled_sequence'].apply(tuple)
unique_missing_patterns = unhandled_sequences['sequence_tuple'].value_counts().reset_index()
unique_missing_patterns.columns = ['sequence_pattern', 'number_of_occurrences']

# 4. Print the results to the console
print("Here are the sequences that resulted in 'None':")
print(unique_missing_patterns)

Here are the sequences that resulted in 'None':
Empty DataFrame
Columns: [sequence_pattern, number_of_occurrences]
Index: []


In [46]:
final_df[final_df['final_resolved']=='Re-Check'][['combined_cow', 'breed_event_id', 'Result', 'final_resolved']]

,combined_cow,breed_event_id,Result,final_resolved


In [48]:
final_df['Result'].value_counts()

Result
Pregnant    36213
Open         8134
Re-Check     3349
Name: count, dtype: int64

In [47]:
final_df.to_csv('3_11_2.csv')

In [16]:
import pandas as pd 

class_df = pd.read_csv('/home/rajesh/work/ornella/3_9_2026.csv')

In [17]:
drop_df= class_df[class_df['breed_event_id']=='20215.0401430.0_2025_7_1']

In [23]:
class_df = class_df.drop(drop_df.index,axis=0)

In [24]:
import pandas as pd

# Load data and ensure datetime format
class_df = pd.read_csv('/home/rajesh/work/ornella/3_9_2026.csv')
class_df['adj_final_breeding'] = pd.to_datetime(class_df['adj_final_breeding'])

# Initialize the column to avoid any assignment issues later
class_df['final_resolved'] = None

def resolve_cow_history(cow_data):
    """
    Processes all breeding events for a single cow, resolving the pregnancy test 
    sequences and adjusting outcomes based on gaps between rebreds.
    """
    # Create a copy to avoid SettingWithCopyWarning, then sort chronologically
    cow_data = cow_data.copy()
    cow_data = cow_data.sort_values(['adj_final_breeding', 'sample_id'])

    # ✅ FIX: Sort breed_events by actual chronological date, not arbitrary .unique() order
    breed_event_dates = cow_data.groupby('breed_event_id')['adj_final_breeding'].min()
    breed_events = breed_event_dates.sort_values().index.tolist()
    n_events = len(breed_events)

    for i, event_id in enumerate(breed_events):
        mask = cow_data['breed_event_id'] == event_id
        event_df = cow_data[mask]

        samples = event_df['Result'].tolist()
        n_samples = len(samples)

        # Skip if no samples exist for this event
        if n_samples == 0:
            continue

        first = samples[0]
        last = samples[-1]
        resolved = None

        # =======================================================
        # STEP 1: Base Resolution (Pattern-Matching)
        # =======================================================
        if n_samples == 1:
            resolved = first
        else:
            # 1. Sequences ending in PREGNANT
            if last == 'Pregnant':
                if first == 'Pregnant':
                    if 'Open' in samples:
                        resolved = 'preg_open_preg'
                    elif 'Re-Check' in samples:
                        resolved = 'Pregant_with_rc'
                    else:
                        resolved = 'Pregnant'
                elif first == 'Re-Check':
                    resolved = 'rc_preg'
                elif first == 'Open':
                    resolved = 'open_to_preg'

            # 2. Sequences ending in OPEN
            elif last == 'Open':
                if 'Pregnant' in samples:
                    resolved = 'loss'
                elif first == 'Re-Check':
                    resolved = 'rc_open'
                else:
                    resolved = 'Open'

            # 3. Sequences ending in RE-CHECK
            elif last == 'Re-Check':
                if 'Pregnant' in samples:
                    resolved = 'preg_to_rc_unresolved'
                elif 'Open' in samples:
                    resolved = 'open_to_rc_unresolved'
                else:
                    resolved = 'rc_unresolved'

        # =======================================================
        # STEP 2: Sequence/Gap Overrides (Rebred Logic)
        # =======================================================
        if n_events == 1:
            # Scenario A: Only one AI event for this cow overall
            if last == 'Re-Check':
                resolved = 'rc_unresolved'
            elif last == 'Open' and 'Pregnant' in samples:
                resolved = 'loss_2_open'

        else:
            # Scenario B: Multiple AI events
            if i < n_events - 1:
                # Not the last event — check gap to next breeding event
                curr_date = event_df['adj_final_breeding'].iloc[0]
                next_event_df = cow_data[cow_data['breed_event_id'] == breed_events[i + 1]]
                next_date = next_event_df['adj_final_breeding'].iloc[0]
                gap = (next_date - curr_date).days

                if gap < 300:
                    if last != 'Open':
                        if all(r == 'Pregnant' for r in samples):
                            resolved = 'preg_loss_rebred'
                        elif 'Pregnant' in samples and 'Re-Check' in samples:
                            resolved = 'rc_preg_loss_rebred'
                        elif all(r == 'Re-Check' for r in samples):
                            resolved = 'rc_rebred'
                        # ✅ FIX: single Re-Check sample rebred within 300 days
                        elif n_samples == 1 and last == 'Re-Check':
                            resolved = 'rc_rebred'
                        # ✅ FIX: single Pregnant sample rebred within 300 days
                        elif n_samples == 1 and last == 'Pregnant':
                            resolved = 'preg_loss_rebred'

                elif gap >= 300:
                    if last == 'Re-Check':
                        resolved = 'rc_unresolved_>_1_bred'

            else:
                # ✅ FIX: Last event in a multi-event cow — was completely unhandled before
                if last == 'Re-Check':
                    resolved = 'rc_unresolved_>_1_bred'
                elif last == 'Open' and 'Pregnant' in samples:
                    resolved = 'loss_2_open'

        # Assign the final computed status back to this specific event's rows
        cow_data.loc[mask, 'final_resolved'] = resolved

    return cow_data


# Apply the unified function to each cow
# group_keys=False prevents pandas from adding an extra index level
final_df = class_df.groupby('combined_cow', group_keys=False).apply(resolve_cow_history)

# Optional: verify your outputs
# print(final_df[['combined_cow', 'breed_event_id', 'Result', 'final_resolved']].head(20))

/tmp/ipykernel_330263/587925458.py:130: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_df = class_df.groupby('combined_cow', group_keys=False).apply(resolve_cow_history)


In [27]:
final_df['final_resolved'].value_counts()

final_resolved
Pregnant                  31439
Open                       6574
Pregant_with_rc            2863
loss                       1438
preg_loss_rebred           1437
rc_open                    1298
rc_preg                     985
loss_2_open                 645
rc_rebred                   358
rc_unresolved_>_1_bred      313
rc_preg_loss_rebred         219
preg_open_preg               74
rc_unresolved                48
open_to_preg                  5
Name: count, dtype: int64

In [ ]:
import pandas as pd

# sort dataset
df = final_df.sort_values(["combined_cow", "DSLB"])

resolved_states = ["Pregnant", "Open", "preg_loss"]

results = []

for cow, g in df.groupby("combined_cow"):

    g = g.reset_index(drop=True)

    recheck_idx = g.index[g["Result"] == "recheck"]

    if len(recheck_idx) == 0:
        continue

    first_recheck = recheck_idx[0]

    # find first resolution after recheck
    resolution = g.loc[first_recheck+1:]
    resolution = resolution[resolution["status"].isin(resolved_states)]

    if len(resolution) == 0:
        continue

    resolution_row = resolution.iloc[0]

    results.append({
        "cow_id": cow,
        "first_recheck_DSLB": g.loc[first_recheck, "DSLB"],
        "resolution_DSLB": resolution_row["DSLB"],
        "days_to_resolution": resolution_row["DSLB"] - g.loc[first_recheck, "DSLB"],
        "samples_to_resolution": resolution_row.name - first_recheck + 1,
        "final_status": resolution_row["status"],
    })

resolution_df = pd.DataFrame(results)

KeyError: 'status'